from utils import setup_korean_font
setup_korean_font()
# 02 선형·생성적 베이스라인 — Phase 3

**전처리:** StandardScaler + SMOTE (LR/LDA/QDA) | StandardScaler + class_weight (SVM)  
**데이터:** `labeled_data.csv` (7,996행, 25개 유효 변수, 불량률 0.89%)  
**CV:** Stratified 5-fold (공유 인덱스)  
**모델:** LR(L1), LR(L2), LDA, QDA, SVM(linear), SVM(RBF)  
**가이드북 비교:** §2.3의 SVM best 결과와 나란히 비교

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

from utils import set_seed
from data import load_raw, get_fold
from preprocess import get_feature_cols

set_seed(42)

FIGURES_DIR = PROJECT_ROOT / 'results' / 'figures'
TABLES_DIR  = PROJECT_ROOT / 'results' / 'tables'

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print('Setup OK')

---
## Cell 2 — 전체 그리드 결과

In [ ]:
full_df = pd.read_csv(TABLES_DIR / 'baseline_grid_full.csv')
summary = pd.read_csv(TABLES_DIR / 'baseline_results.csv')

print(f'총 실험 수: {len(full_df)} (6 모델 × 하이퍼파라미터 그리드 × 5-fold)')
print()

disp_cols = ['family', 'roc_auc_mean', 'roc_auc_std', 'pr_auc_mean', 'pr_auc_std', 'f1_mean', 'f1_std']
print('=== 전체 그리드 결과 (ROC-AUC 정렬) ===')
print(full_df[disp_cols].sort_values('roc_auc_mean', ascending=False).dropna().to_string(index=False))

---
## Cell 3 — 모델별 Best 결과표 (가이드북 비교)

In [ ]:
print('=== Phase 3 Best-per-Family 결과표 ===')
print('(가이드북 §2.3 SVM 결과는 원본 PDF 참조 — 직접 수치 없음)')
print()

display = summary[['model','preprocessing',
                   'roc_auc_mean','roc_auc_std',
                   'pr_auc_mean', 'pr_auc_std',
                   'f1_mean','f1_std',
                   'guidebook_note']].copy()

display['ROC-AUC'] = display.apply(
    lambda r: f"{r['roc_auc_mean']:.4f} +/- {r['roc_auc_std']:.4f}", axis=1)
display['PR-AUC']  = display.apply(
    lambda r: f"{r['pr_auc_mean']:.4f} +/- {r['pr_auc_std']:.4f}", axis=1)

print(display[['model','ROC-AUC','PR-AUC','guidebook_note']].to_string(index=False))

best_roc = summary.loc[summary['roc_auc_mean'].idxmax()]
best_pr  = summary.loc[summary['pr_auc_mean'].idxmax()]
print(f'\nBest ROC-AUC: {best_roc["model"]} = {best_roc["roc_auc_mean"]:.4f}')
print(f'Best PR-AUC : {best_pr["model"]}  = {best_pr["pr_auc_mean"]:.4f}')

---
## Cell 4 — ROC 곡선 비교

In [ ]:
img = mpimg.imread(str(FIGURES_DIR / 'baseline_roc_curves.png'))
fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'baseline_roc_curves.png', bbox_inches='tight')
plt.show()

6개 모델 모두 ROC-AUC 0.90 이상을 기록했으며, QDA가 0.9344로 최고였다. SVM-RBF도 0.9297로 경쟁력 있는 수준이다. 그러나 ROC-AUC만 놓고 보면 선형 결정경계(LR-L2 0.9343)와 QDA·SVM-RBF의 격차가 0.001 수준에 불과해, 선형 모델 자체가 이미 충분히 강력하다는 점을 확인했다.

---
## Cell 5 — PR 곡선 비교

In [ ]:
img = mpimg.imread(str(FIGURES_DIR / 'baseline_pr_curves.png'))
fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

PR-AUC 순위는 QDA(0.37) > SVM-linear(0.34) > LR(0.26) > LDA(0.13) > SVM-RBF(0.08)이다. SVM-RBF는 ROC-AUC에서 상위권이지만 PR-AUC가 최저인데, 극단적 불균형 데이터에서 높은 recall 대비 precision이 급락하기 때문이다. QDA가 클래스별 공분산을 독립적으로 모델링한 덕분에 불량 패턴 포착에서 유리한 위치를 점한다.

---
## Cell 6 — 막대 비교 + 결론

In [ ]:
img = mpimg.imread(str(FIGURES_DIR / 'baseline_comparison_bar.png'))
fig, ax = plt.subplots(figsize=(14, 5))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

print('=' * 65)
print('Phase 3 선형 베이스라인 결론')
print('=' * 65)
print("""
[핵심 발견]
1. ROC-AUC 기준: QDA(0.9344) == LR-L1/L2(~0.9342) >> SVM
   -> 선형 결정경계로 ROC-AUC 0.93+ 달성 가능 (강력한 선형 베이스라인)

2. PR-AUC 기준: QDA(0.35) > SVM-linear(0.33) > LR(0.26) > LDA(0.13) > SVM-RBF(0.08)
   -> QDA의 클래스별 공분산이 불량 샘플 분포 포착에 효과적
   -> SVM-RBF는 ROC-AUC 높지만 PR-AUC 최저 (임계값 0.5에서 예측 편향)

3. 가이드북 비교:
   - 가이드북 §2.3: SVM이 베스트 → 우리 결과 일부 일치 (SVM-linear PR-AUC 2위)
   - 그러나 QDA가 모든 지표 1위 (가이드북에서 시도하지 않은 모델)
   - '단일 SVM best 선택'보다 ablation이 QDA라는 숨겨진 강자 발굴

4. Phase 4 진입 후보:
   - 앙상블/NN 비교 기준: QDA(PR-AUC 0.35), LR-L2(ROC-AUC 0.93)
   - SVM-RBF는 Phase 5 Stacking base learner로 포함 (ROC-AUC 기여)
""")

선형 결정경계(LR-L2)만으로 ROC-AUC 0.9343을 달성했다. Phase 4 앙상블·NN이 이 수치를 유의미하게 넘어서는지가 다음 단계의 핵심 검증 포인트다. 가이드북이 시도하지 않은 모델 중에서는 QDA(PR-AUC 0.35)가 최고 성능을 기록해, 단순 SVM 단일 선택보다 ablation이 더 강한 후보를 발굴한다는 점을 보였다.